# Fintech AML Intelligence System - Transaction Risk Monitoring & SAR Analytics

End-to-end AML transaction monitoring system covering data cleaning, feature engineering,
SQL analysis, machine learning fraud detection, and Power BI visualization.

**Author:** Emmanuel Bitrus  
**Tools:** Python (pandas, scikit-learn) · SQL (SQLite) · Power BI

# Section 1

## Phase 1: Data Cleaning

Loading the raw synthetic AML dataset and resolving data quality issues -
missing values, duplicates, inconsistent data types, and dirty currency strings.

In [288]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [289]:
import pandas as pd

import numpy as np

df = pd.read_excel('/content/drive/MyDrive/AML_Dataset_Raw.xlsx')

In [290]:
df.info() #checking the datatype of each header (transaction_amount should be float)
          # (transaction date should be date-time, and
          # transaction time should be converted to time, though still object)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Transaction_ID      508 non-null    object
 1   Customer_ID         508 non-null    object
 2   Customer_Name       508 non-null    object
 3   Account_Type        508 non-null    object
 4   Transaction_Amount  493 non-null    object
 5   Transaction_Date    508 non-null    object
 6   Transaction_Time    508 non-null    object
 7   Sender_Country      508 non-null    object
 8   Receiver_Country    496 non-null    object
 9   Transaction_Type    508 non-null    object
 10  Frequency_7days     508 non-null    int64 
 11  Previous_SAR_Flag   508 non-null    int64 
 12  AML_Label           508 non-null    int64 
dtypes: int64(3), object(10)
memory usage: 51.7+ KB


In [291]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nData types:\n", df.dtypes)
print("\nAccount_Type values:\n", df['Account_Type'].value_counts())
print("\nDuplicates:", df.duplicated().sum())

Shape: (508, 13)

Missing values:
 Transaction_ID         0
Customer_ID            0
Customer_Name          0
Account_Type           0
Transaction_Amount    15
Transaction_Date       0
Transaction_Time       0
Sender_Country         0
Receiver_Country      12
Transaction_Type       0
Frequency_7days        0
Previous_SAR_Flag      0
AML_Label              0
dtype: int64

Data types:
 Transaction_ID        object
Customer_ID           object
Customer_Name         object
Account_Type          object
Transaction_Amount    object
Transaction_Date      object
Transaction_Time      object
Sender_Country        object
Receiver_Country      object
Transaction_Type      object
Frequency_7days        int64
Previous_SAR_Flag      int64
AML_Label              int64
dtype: object

Account_Type values:
 Account_Type
Personal    253
Business    250
personal      3
business      2
Name: count, dtype: int64

Duplicates: 7


In [292]:
df.shape

(508, 13)

In [293]:
df.describe()

,Frequency_7days,Previous_SAR_Flag,AML_Label
count,508.000000,508.000000,508.000000
mean,7.120079,0.226378,0.401575
std,5.453348,0.418899,0.490700
min,1.000000,0.000000,0.000000
25%,3.000000,0.000000,0.000000
50%,6.000000,0.000000,0.000000
75%,10.000000,0.000000,1.000000
max,25.000000,1.000000,1.000000


In [294]:
df.count()

,0
Transaction_ID,508
Customer_ID,508
Customer_Name,508
Account_Type,508
Transaction_Amount,493
Transaction_Date,508
Transaction_Time,508
Sender_Country,508
Receiver_Country,496
Transaction_Type,508


In [295]:
df.isnull().sum() #showing how many values are null, missing, or inproper

,0
Transaction_ID,0
Customer_ID,0
Customer_Name,0
Account_Type,0
Transaction_Amount,15
Transaction_Date,0
Transaction_Time,0
Sender_Country,0
Receiver_Country,12
Transaction_Type,0


In [296]:
df['Transaction_Amount'] = pd.to_numeric(df['Transaction_Amount'], errors='coerce')
df['Transaction_Amount'] = df['Transaction_Amount'].fillna(df['Transaction_Amount'].median())

In [297]:
df.isnull().sum() #after filling missing data (transaction amount to median)
                  # (missing receiver country to unknown)

,0
Transaction_ID,0
Customer_ID,0
Customer_Name,0
Account_Type,0
Transaction_Amount,0
Transaction_Date,0
Transaction_Time,0
Sender_Country,0
Receiver_Country,12
Transaction_Type,0


In [298]:
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])

df['Transaction_Time'] = pd.to_datetime(df['Transaction_Time'], format='%H:%M').dt.time

In [299]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Transaction_ID      508 non-null    object        
 1   Customer_ID         508 non-null    object        
 2   Customer_Name       508 non-null    object        
 3   Account_Type        508 non-null    object        
 4   Transaction_Amount  508 non-null    float64       
 5   Transaction_Date    508 non-null    datetime64[ns]
 6   Transaction_Time    508 non-null    object        
 7   Sender_Country      508 non-null    object        
 8   Receiver_Country    496 non-null    object        
 9   Transaction_Type    508 non-null    object        
 10  Frequency_7days     508 non-null    int64         
 11  Previous_SAR_Flag   508 non-null    int64         
 12  AML_Label           508 non-null    int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object

In [300]:
df.duplicated().sum()

df[df.duplicated(keep=False)]

# 8 duplicates found

,Transaction_ID,Customer_ID,Customer_Name,Account_Type,Transaction_Amount,Transaction_Date,Transaction_Time,Sender_Country,Receiver_Country,Transaction_Type,Frequency_7days,Previous_SAR_Flag,AML_Label
37,TXN1039,C006,Fatima Bello,Business,430.45,2026-05-30,17:09:00,Kenya,Nigeria,Cash Deposit,3,0,0
68,TXN1300,C007,Tunde Adeleke,Personal,1219.23,2026-01-22,03:53:00,Kenya,Kenya,Cash Deposit,4,0,0
69,TXN1334,C009,Ibrahim Musa,Personal,6253.86,2026-06-03,19:46:00,Nigeria,NaN,Cash Deposit,2,0,0
76,TXN1039,C006,Fatima Bello,Business,430.45,2026-05-30,17:09:00,Kenya,Nigeria,Cash Deposit,3,0,0
159,TXN1427,C007,Tunde Adeleke,Personal,42072.97,2026-05-09,18:19:00,Nigeria,NaN,Wire Transfer,7,1,1
197,TXN1187,C005,Emeka Nwosu,Personal,100000.00,2026-01-10,20:12:00,South Africa,Nigeria,Cash Deposit,10,0,1
200,TXN1193,C001,Adewale Okafor,Personal,2993.99,2026-02-02,02:15:00,South Africa,USA,Cash Deposit,1,0,0
302,TXN1334,C009,Ibrahim Musa,Personal,6253.86,2026-06-03,19:46:00,Nigeria,NaN,Cash Deposit,2,0,0
392,TXN1300,C007,Tunde Adeleke,Personal,1219.23,2026-01-22,03:53:00,Kenya,Kenya,Cash Deposit,4,0,0
436,TXN1427,C007,Tunde Adeleke,Personal,42072.97,2026-05-09,18:19:00,Nigeria,NaN,Wire Transfer,7,1,1


In [301]:
df = df.drop_duplicates()

df.duplicated().sum()

np.int64(0)

In [302]:
#  checking for capitalization errors (Account Type has some errors)

print(

    [df['Account_Type'].value_counts()],
    [df['Receiver_Country'].value_counts()],
    [df['Sender_Country'].value_counts()]

    )



[Account_Type
Business    249
Personal    247
personal      3
business      2
Name: count, dtype: int64] [Receiver_Country
UK              87
USA             77
Kenya           73
Nigeria         72
Ghana           71
UAE             27
North Korea     20
Iran            19
Myanmar         18
Russia          16
South Africa    11
Name: count, dtype: int64] [Sender_Country
Nigeria         236
South Africa     88
Kenya            83
Ghana            76
USA               8
UAE               6
UK                4
Name: count, dtype: int64]


In [303]:
df['Account_Type'] = df['Account_Type'].str.title()

df['Account_Type'].value_counts()

,count
Account_Type,
Business,251
Personal,250


In [304]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nData types:\n", df.dtypes)
print("\nAccount_Type values:\n", df['Account_Type'].value_counts())
print("\nDuplicates:", df.duplicated().sum())

Shape: (501, 13)

Missing values:
 Transaction_ID         0
Customer_ID            0
Customer_Name          0
Account_Type           0
Transaction_Amount     0
Transaction_Date       0
Transaction_Time       0
Sender_Country         0
Receiver_Country      10
Transaction_Type       0
Frequency_7days        0
Previous_SAR_Flag      0
AML_Label              0
dtype: int64

Data types:
 Transaction_ID                object
Customer_ID                   object
Customer_Name                 object
Account_Type                  object
Transaction_Amount           float64
Transaction_Date      datetime64[ns]
Transaction_Time              object
Sender_Country                object
Receiver_Country              object
Transaction_Type              object
Frequency_7days                int64
Previous_SAR_Flag              int64
AML_Label                      int64
dtype: object

Account_Type values:
 Account_Type
Business    251
Personal    250
Name: count, dtype: int64

Duplicates: 0


# Section 2

## Phase 2: Feature Engineering

Engineering AML red flag features from raw transaction data:
- Customer behavioral baseline (mean and standard deviation per customer)
- Velocity_Flag: deviation from customer's own normal activity
- Structuring_Flag: regulatory (CBN/NFIU) and policy-based thresholds
- Geo_Risk_Flag: transactions to FATF-designated high-risk jurisdictions
- Round_Number_Flag: suspiciously clean transaction amounts
- Previous_SAR_Flag: prior suspicious activity history
- AML_Risk_Score: composite weighted risk score

In [305]:
df.to_csv('/AML_Dataset_Clean_csv', index=False)

In [306]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/AML_Dataset_Clean_csv')

In [307]:
df = df.sort_values(['Customer_ID', 'Transaction_Date'])
df = df.reset_index(drop=True)
df

,Transaction_ID,Customer_ID,Customer_Name,Account_Type,Transaction_Amount,Transaction_Date,Transaction_Time,Sender_Country,Receiver_Country,Transaction_Type,AML_Label
0,TXN1231,C001,Adewale Okafor,Personal,2190.78,2026-01-21,01:41:00,Ghana,USA,POS Transfer,0
1,TXN1166,C001,Adewale Okafor,Personal,3936.73,2026-01-21,08:28:00,South Africa,UK,Mobile Money,0
2,TXN1310,C001,Adewale Okafor,Personal,9475.53,2026-01-24,12:24:00,Nigeria,Iran,Wire Transfer,1
3,TXN1336,C001,Adewale Okafor,Personal,4198.50,2026-01-26,08:14:00,Ghana,UK,Mobile Money,0
4,TXN1052,C001,Adewale Okafor,Personal,7870.05,2026-01-30,08:11:00,Nigeria,Nigeria,POS Transfer,0
...,...,...,...,...,...,...,...,...,...,...,...
495,TXN1035,C010,Sandra Uchenna,Business,3458.18,2026-06-02,01:39:00,Nigeria,UAE,Mobile Money,1
496,TXN1294,C010,Sandra Uchenna,Business,9796.60,2026-06-04,01:55:00,Nigeria,UK,Wire Transfer,1
497,TXN1326,C010,Sandra Uchenna,Business,1870.96,2026-06-07,06:29:00,Nigeria,UK,Mobile Money,1
498,TXN1080,C010,Sandra Uchenna,Business,5380.23,2026-06-10,07:06:00,Nigeria,USA,Wire Transfer,0


In [308]:
df = df.sort_values(['Customer_ID', 'Transaction_Date'])
df = df.reset_index(drop=True)
df

,Transaction_ID,Customer_ID,Customer_Name,Account_Type,Transaction_Amount,Transaction_Date,Transaction_Time,Sender_Country,Receiver_Country,Transaction_Type,AML_Label
0,TXN1231,C001,Adewale Okafor,Personal,2190.78,2026-01-21,01:41:00,Ghana,USA,POS Transfer,0
1,TXN1166,C001,Adewale Okafor,Personal,3936.73,2026-01-21,08:28:00,South Africa,UK,Mobile Money,0
2,TXN1310,C001,Adewale Okafor,Personal,9475.53,2026-01-24,12:24:00,Nigeria,Iran,Wire Transfer,1
3,TXN1336,C001,Adewale Okafor,Personal,4198.50,2026-01-26,08:14:00,Ghana,UK,Mobile Money,0
4,TXN1052,C001,Adewale Okafor,Personal,7870.05,2026-01-30,08:11:00,Nigeria,Nigeria,POS Transfer,0
...,...,...,...,...,...,...,...,...,...,...,...
495,TXN1035,C010,Sandra Uchenna,Business,3458.18,2026-06-02,01:39:00,Nigeria,UAE,Mobile Money,1
496,TXN1294,C010,Sandra Uchenna,Business,9796.60,2026-06-04,01:55:00,Nigeria,UK,Wire Transfer,1
497,TXN1326,C010,Sandra Uchenna,Business,1870.96,2026-06-07,06:29:00,Nigeria,UK,Mobile Money,1
498,TXN1080,C010,Sandra Uchenna,Business,5380.23,2026-06-10,07:06:00,Nigeria,USA,Wire Transfer,0


In [309]:
# First calculate daily transaction count per customer
df['Daily_Tx_Count'] = df.groupby(
    ['Customer_ID', 'Transaction_Date']
)['Transaction_ID'].transform('count')

# Calculate daily transaction total amount per customer
df['Daily_Tx_Amount'] = df.groupby(
    ['Customer_ID', 'Transaction_Date']
)['Transaction_Amount'].transform('sum')

In [310]:
# Calculate each customer's personal baseline
Customer_Baseline = df.groupby('Customer_ID').agg(
    Avg_Daily_Amount  = ('Transaction_Amount', 'mean'),
    Std_Daily_Amount  = ('Transaction_Amount', 'std'),
    Avg_Daily_Count   = ('Daily_Tx_Count', 'mean'),
    Std_Daily_Count   = ('Daily_Tx_Count', 'std')
).reset_index()

In [311]:
df = df.merge(Customer_Baseline, on='Customer_ID', how='left')

In [312]:
df['Velocity_Flag'] = (
    (df['Daily_Tx_Amount'] > df['Avg_Daily_Amount'] + 2 * df['Std_Daily_Amount']) |
    (df['Daily_Tx_Count']  > df['Avg_Daily_Count']  + 2 * df['Std_Daily_Count'])
).astype(int)

In [313]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Transaction_ID      500 non-null    object 
 1   Customer_ID         500 non-null    object 
 2   Customer_Name       500 non-null    object 
 3   Account_Type        500 non-null    object 
 4   Transaction_Amount  500 non-null    float64
 5   Transaction_Date    500 non-null    object 
 6   Transaction_Time    500 non-null    object 
 7   Sender_Country      500 non-null    object 
 8   Receiver_Country    500 non-null    object 
 9   Transaction_Type    500 non-null    object 
 10  AML_Label           500 non-null    int64  
 11  Daily_Tx_Count      500 non-null    int64  
 12  Daily_Tx_Amount     500 non-null    float64
 13  Avg_Daily_Amount    500 non-null    float64
 14  Std_Daily_Amount    500 non-null    float64
 15  Avg_Daily_Count     500 non-null    float64
 16  Std_Dail

In [314]:
cols_to_drop = [col for col in df.columns if col.endswith('_x') or col.endswith('_y')]
df = df.drop(columns=cols_to_drop)

In [315]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Transaction_ID      500 non-null    object 
 1   Customer_ID         500 non-null    object 
 2   Customer_Name       500 non-null    object 
 3   Account_Type        500 non-null    object 
 4   Transaction_Amount  500 non-null    float64
 5   Transaction_Date    500 non-null    object 
 6   Transaction_Time    500 non-null    object 
 7   Sender_Country      500 non-null    object 
 8   Receiver_Country    500 non-null    object 
 9   Transaction_Type    500 non-null    object 
 10  AML_Label           500 non-null    int64  
 11  Daily_Tx_Count      500 non-null    int64  
 12  Daily_Tx_Amount     500 non-null    float64
 13  Avg_Daily_Amount    500 non-null    float64
 14  Std_Daily_Amount    500 non-null    float64
 15  Avg_Daily_Count     500 non-null    float64
 16  Std_Dail

In [316]:
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])

In [317]:
df['Velocity_Flag'] = (
    (df['Daily_Tx_Amount'] > df['Avg_Daily_Amount'] + 2 * df['Std_Daily_Amount']) |
    (df['Daily_Tx_Count']  > df['Avg_Daily_Count']  + 2 * df['Std_Daily_Count'])
).astype(int)

In [318]:
df['Velocity_Flag'].value_counts()

,count
Velocity_Flag,
0,442
1,58


In [319]:
# Layer 1  Regulatory (CBN/NFIU) Structuring Flag
df['Regulatory_Structuring_Flag'] = (
    ((df['Account_Type'] == 'Personal') &
    (df['Daily_Tx_Amount'] >= 4000000) &
    (df['Daily_Tx_Amount'] < 5000000)) |
    ((df['Account_Type'] == 'Businesss') &
    (df['Daily_Tx_Amount'] >= 9000000) &
    (df['Daily_Tx_Amount'] < 10000000))
).astype(int)

# Layer 2  Policy (Internal) Structuring Flag
df['Policy_Structuring_Flag'] = (
    ((df['Account_Type'] == 'Personal') &
    (df['Daily_Tx_Amount'] >= 9000) &
    (df['Daily_Tx_Amount'] < 9999)) |
    ((df['Account_Type'] == 'Businesss') &
    (df['Daily_Tx_Amount'] >= 19000) &
    (df['Daily_Tx_Amount'] < 19999))
).astype(int)

# Combined Structuring Flag  either basis triggers it
df['Structuring_Flag'] = (
    (df['Regulatory_Structuring_Flag'] == 1) | (df['Policy_Structuring_Flag'] == 1)
    ).astype(int)


In [320]:
print("Regulatory:", df['Regulatory_Structuring_Flag'].value_counts().to_dict())
print("Policy:", df['Policy_Structuring_Flag'].value_counts().to_dict())
print("Combined:", df['Structuring_Flag'].value_counts().to_dict())

Regulatory: {0: 500}
Policy: {0: 466, 1: 34}
Combined: {0: 466, 1: 34}


In [321]:
df['Receiver_Country'].unique()

array(['USA', 'UK', 'Iran', 'Nigeria', 'Unknown', 'Myanmar',
       'South Africa', 'Ghana', 'North Korea', 'Kenya', 'Russia', 'UAE'],
      dtype=object)

In [322]:
High_Risk_Countries = ['Russia', 'Iran', 'North Korea', 'Myanmar']

df['Geo_Risk_Flag'] = df['Receiver_Country'].isin(High_Risk_Countries).astype(int)

df['Geo_Risk_Flag'].value_counts()






,count
Geo_Risk_Flag,
0,419
1,81


In [323]:
df['Round_Number_Flag'] = (df['Transaction_Amount'] % 1000 == 0).astype(int)

df['Round_Number_Flag'].value_counts()

,count
Round_Number_Flag,
0,449
1,51


In [324]:
df['Previous_SAR_Flag'] = df.groupby('Customer_ID')['AML_Label'].shift(1).fillna(0).astype(int)

df['Previous_SAR_Flag'].value_counts()

,count
Previous_SAR_Flag,
0,302
1,198


In [325]:
df['AML_Risk_Score'] = (
    (df['Geo_Risk_Flag']      * 0.35) +
    (df['Structuring_Flag']   * 0.25) +
    (df['Velocity_Flag']      * 0.20) +
    (df['Previous_SAR_Flag']  * 0.15) +
    (df['Round_Number_Flag']  * 0.05)
)

df['AML_Risk_Score'].describe()

,AML_Risk_Score
count,500.000000
mean,0.161400
std,0.192286
min,0.000000
25%,0.000000
50%,0.150000
75%,0.200000
max,0.750000


In [326]:
df[df['AML_Risk_Score'] >= 0.50][['Customer_ID', 'Customer_Name', 'Transaction_Amount', 'AML_Risk_Score', 'AML_Label']].sort_values('AML_Risk_Score', ascending=False)

,Customer_ID,Customer_Name,Transaction_Amount,AML_Risk_Score,AML_Label
22,C001,Adewale Okafor,9430.26,0.75,1
146,C003,Mohammed Al-Rashid,9730.03,0.75,1
21,C001,Adewale Okafor,9814.59,0.75,1
217,C005,Emeka Nwosu,9276.21,0.75,1
341,C007,Tunde Adeleke,9453.05,0.75,1
352,C007,Tunde Adeleke,9441.34,0.75,1
224,C005,Emeka Nwosu,9635.98,0.75,1
337,C007,Tunde Adeleke,9753.66,0.75,1
86,C002,Ngozi Eze,9100.93,0.70,1
202,C004,Chidinma Obi,22628.80,0.70,1


In [327]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Transaction_ID               500 non-null    object        
 1   Customer_ID                  500 non-null    object        
 2   Customer_Name                500 non-null    object        
 3   Account_Type                 500 non-null    object        
 4   Transaction_Amount           500 non-null    float64       
 5   Transaction_Date             500 non-null    datetime64[ns]
 6   Transaction_Time             500 non-null    object        
 7   Sender_Country               500 non-null    object        
 8   Receiver_Country             500 non-null    object        
 9   Transaction_Type             500 non-null    object        
 10  AML_Label                    500 non-null    int64         
 11  Daily_Tx_Count               500 non-null    

# Section 3

## Phase 3: Export for SQL Analysis

Saving the fully featured dataset to SQLite for compliance-focused SQL querying.
SQL queries are documented separately in `AML_Dataset_Session.sql`.



In [328]:
import sqlite3

conn = sqlite3.connect('AML_Database.db')
df.to_sql('aml_transactions', conn, if_exists='replace', index=False)

# Verify the table was created
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM aml_transactions")
print("Rows in table:", cursor.fetchone()[0])

conn.commit()
conn.close()
print("Database saved successfully")

Rows in table: 500
Database saved successfully


# Section 4

## Phase 4: Machine Learning: Fraud Detection Model

Training and comparing Logistic Regression and Random Forest models to predict
suspicious transactions, with emphasis on recall given the asymmetric cost of
false negatives in AML compliance.

In [329]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/drive/MyDrive/AML_Feature_Engineered_Dataset.csv')

# Define target variable y
y = df['AML_Label']

# Define features X by dropping non-feature columns and the target column
X = df.drop(columns=[
'Transaction_ID',
    'Customer_ID',
    'Customer_Name',
    'Transaction_Date',
    'Transaction_Time',
    'Sender_Country',
    'Receiver_Country',
    'AML_Label'  # Dropping for simplicity, could be engineered further
])

# Identify categorical columns for one-hot encoding
categorical_cols = X.select_dtypes(include='object').columns

# Apply one-hot encoding to categorical features
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

X = X.drop(columns=['AML_Risk_Score', 'Regulatory_Structuring_Flag', 'Policy_Structuring_Flag'])

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



print(X.shape)
X.columns

(500, 16)


Index(['Transaction_Amount', 'Daily_Tx_Count', 'Daily_Tx_Amount',
       'Avg_Daily_Amount', 'Std_Daily_Amount', 'Avg_Daily_Count',
       'Std_Daily_Count', 'Velocity_Flag', 'Structuring_Flag', 'Geo_Risk_Flag',
       'Round_Number_Flag', 'Previous_SAR_Flag', 'Account_Type_Personal',
       'Transaction_Type_Mobile Money', 'Transaction_Type_POS Transfer',
       'Transaction_Type_Wire Transfer'],
      dtype='object')

In [330]:
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (400, 16)
Testing set: (100, 16)


### Model 1: Logistic Regression (Baseline)

In [331]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

In [332]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [333]:
y_pred_lr = lr_model.predict(X_test_scaled)

In [334]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

[[55  1]
 [13 31]]
              precision    recall  f1-score   support

           0       0.81      0.98      0.89        56
           1       0.97      0.70      0.82        44

    accuracy                           0.86       100
   macro avg       0.89      0.84      0.85       100
weighted avg       0.88      0.86      0.86       100



### Model 2: Random Forest

Comparing against Logistic Regression baseline.

In [335]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

In [336]:
y_pred_rf = rf_model.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[[54  2]
 [11 33]]
              precision    recall  f1-score   support

           0       0.83      0.96      0.89        56
           1       0.94      0.75      0.84        44

    accuracy                           0.87       100
   macro avg       0.89      0.86      0.86       100
weighted avg       0.88      0.87      0.87       100



### Feature Importance Analysis

Identifying which features the Random Forest model relied on most heavily.

In [337]:


feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(feature_importance)

                           Feature  Importance
0               Transaction_Amount    0.337544
2                  Daily_Tx_Amount    0.223419
9                    Geo_Risk_Flag    0.124975
10               Round_Number_Flag    0.083015
15  Transaction_Type_Wire Transfer    0.061970
1                   Daily_Tx_Count    0.024035
14   Transaction_Type_POS Transfer    0.023188
3                 Avg_Daily_Amount    0.018906
6                  Std_Daily_Count    0.017911
4                 Std_Daily_Amount    0.017037
5                  Avg_Daily_Count    0.015945
13   Transaction_Type_Mobile Money    0.015235
8                 Structuring_Flag    0.014200
11               Previous_SAR_Flag    0.013838
12           Account_Type_Personal    0.005180
7                    Velocity_Flag    0.003601


### Applying the Model - Fraud Probability Scoring

Generating fraud probability scores for all 500 transactions and comparing
against the rules-based AML_Risk_Score.

In [338]:
# Get probability scores for ALL transactions, not just test set
X_all_scaled = scaler.transform(X)
df['ML_Fraud_Probability'] = rf_model.predict_proba(X_all_scaled)[:, 1]

df[['Customer_ID', 'Customer_Name', 'Transaction_Amount',
    'AML_Label', 'ML_Fraud_Probability']].sort_values(
    'ML_Fraud_Probability', ascending=False).head(10)

,Customer_ID,Customer_Name,Transaction_Amount,AML_Label,ML_Fraud_Probability
478,C010,Sandra Uchenna,10000.00,1,1.0
476,C010,Sandra Uchenna,9327.93,1,1.0
468,C010,Sandra Uchenna,33540.71,1,1.0
467,C010,Sandra Uchenna,9225.62,1,1.0
18,C001,Adewale Okafor,9577.34,1,1.0
22,C001,Adewale Okafor,9430.26,1,1.0
21,C001,Adewale Okafor,9814.59,1,1.0
461,C010,Sandra Uchenna,34719.27,1,1.0
443,C009,Ibrahim Musa,46132.20,1,1.0
47,C001,Adewale Okafor,10000.00,1,1.0


In [339]:
# Compare rules-based risk ranking vs ML-based risk ranking
comparison = df.groupby(['Customer_ID', 'Customer_Name']).agg(
    Avg_Rules_Risk_Score=('AML_Risk_Score', 'mean'),
    Avg_ML_Probability=('ML_Fraud_Probability', 'mean')
).round(3).sort_values('Avg_ML_Probability', ascending=False)

print(comparison)

                                Avg_Rules_Risk_Score  Avg_ML_Probability
Customer_ID Customer_Name                                               
C008        Grace Mensah                       0.205               0.525
C007        Tunde Adeleke                      0.225               0.501
C004        Chidinma Obi                       0.160               0.466
C005        Emeka Nwosu                        0.202               0.411
C002        Ngozi Eze                          0.144               0.390
C003        Mohammed Al-Rashid                 0.156               0.380
C009        Ibrahim Musa                       0.145               0.351
C001        Adewale Okafor                     0.156               0.343
C006        Fatima Bello                       0.098               0.323
C010        Sandra Uchenna                     0.143               0.314


In [340]:
comparison.to_csv('Rules_vs_ML_Comparison.csv')
df.to_csv('AML_Dataset_Final_With_ML.csv', index=False)

# Section 5

## Summary

- Random Forest selected as final model (75% recall vs 70% for Logistic Regression)
- Top predictive features: Transaction_Amount, Daily_Tx_Amount, Geo_Risk_Flag
- Engineered behavioral flags (Velocity, Structuring, Previous_SAR) showed lower
  individual importance than raw transaction signals, informing future iterations
- ML and rules-based approaches showed strong directional agreement on highest-risk
  customers (Grace Mensah, Tunde Adeleke), with some divergence on edge cases

**Limitations:** Dataset is synthetic with elevated suspicious rate (40%) to support
model training. Real-world AML suspicious rates typically range 1-5%. Small sample
size (500 rows) limits model generalizability, production systems would require
substantially larger training data.